# Example: Cell Painting Preset

**Before you start:** Run from `Notebooks/examples/`. Replace `path/to/images`, `path/to/masks`, and the metadata template. Cell Painting usually has **one mask per field image**; masks are required for shape and masked-intensity features in typical presets.

Combines deep embeddings with classical image/mask properties. See [Property Reference](../../docs/guides/property-reference.md) and [Property-Based Analysis](../../docs/examples/workflows.md#property-based-analysis).

**Author:** Aitor González-Marfil ([@AAitorG](https://github.com/AAitorG))

---

## 1. Setup

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# import pandas as pd  # uncomment for Option B (CSV metadata) below
from phenome import (
    PhenoMe,
    get_metadata_from_path,
    load_dinov2_model,
    # make_dataframe_metadata_fn,  # uncomment with Option B
)

## 2. Load Data with Masks

Pass `mask_dir` to `find_files` so each image gets a `mask_path`. **Option A** fits when plate/well/treatment are encoded in the folder path. **Option B** fits when a spreadsheet lists filename → treatment (or other columns) and paths do not carry that information. See [Custom Metadata](../../docs/guides/custom-metadata.md) and [mask discovery in find_files](../../docs/guides/custom-metadata.md#mask-discovery-in-find_files).

In [ ]:
pheno = PhenoMe(seed=42)

image_dir = "path/to/images"
mask_dir = "path/to/masks"

# Option A: Path-based metadata
metadata_fn = get_metadata_from_path(".../(treatment)/(filename).*")

# Option B: CSV-based metadata — uncomment imports in the Setup cell, then:
# df_meta = pd.read_csv("metadata.csv")
# metadata_fn = make_dataframe_metadata_fn(df_meta, filename_column="filename")

file_df = pheno.find_files(
    image_dir,
    mask_dir=mask_dir if os.path.exists(mask_dir) else None,
    metadata_fn=metadata_fn,
    extensions=[".tif", ".tiff", ".png"],
)

print(f"Found {len(file_df)} images")

## 3. Process and Compute Properties

Use `property_preset='basic'` or `'full'` for Cell Painting. Presets include regionprops, intensity, masked intensity.

In [ ]:
model, wrapper = load_dinov2_model(model_name="dinov2_vitb14_reg")
pheno.process_images(wrapper, channel_mode="split")

# Compute properties (requires masks for shape-based presets)
df_props = pheno.compute_properties(property_preset="basic")
print(f"Computed {len(df_props.columns)} properties")

## 4. Access Embeddings

Use `get_embeddings()` for lazy-safe access when using checkpoints.

In [ ]:
embeddings = pheno.get_embeddings()
if embeddings is not None:
    print(f"Embeddings shape: {embeddings.shape}")

## 5. Distance Analysis and Visualization

Use `treatment` for grouping when metadata has that key.

In [ ]:
metadata_keys = pheno.get_available_metadata_keys()
group_by = (
    "treatment"
    if "treatment" in metadata_keys
    else "condition"
    if "condition" in metadata_keys
    else metadata_keys[0]
    if metadata_keys
    else None
)

distance_results = pheno.compute_reference_distances(
    reference_filters={group_by: "Control"} if group_by else {"condition": "Control"},
    source="embeddings",
    mode="centroid",
)

pheno.plot_pca(color_by=group_by)
pheno.plot_distance_distribution(distance_results=distance_results, group_by=group_by, kde=True)

## See Also

- [04_exploratory_analysis.ipynb](04_exploratory_analysis.ipynb): Clustering, prototypes, outliers, property explainability
- [example_advanced_phenotyping.py](example_advanced_phenotyping.py): Full script-style pipeline
- [Property Reference](../../docs/guides/property-reference.md): Preset contents
- [Custom Metadata](../../docs/guides/custom-metadata.md): Path templates and CSV lookup